# Pruning Analysis — Visualization and Degradation Figures

Runs entirely on MacBook. No training, no GPU required.

**Section A — Per-label degradation vs prevalence**  
Shows empirically that standard DynamicViT degrades rare classes disproportionately
(Spearman correlation: prevalence vs AUC degradation at 30% keep).
CA-DynamicViT breaks this correlation.

**Section B — Grad-CAM vs token mask visualization**  
Side-by-side figure for paper: original image | Grad-CAM | std_30% mask | CA_30% mask.
Focal pathologies: Pneumonia, Pleural Effusion, Cardiomegaly.

**Files needed from Drive** (download once, then notebook runs offline):
- `checkpoints/token_pruning_auc.csv`
- `checkpoints/ca_vs_standard_auc.csv`
- `checkpoints/pruning_auc_comparison.csv`
- `checkpoints/token_pruning_iou.csv`
- `checkpoints/ca_vs_standard_iou.csv`
- `maps/token_masks/masks_{30,70}pct.npy`
- `maps/token_masks/masks_ca_{30,70}pct.npy`

**Outputs** (all saved to `maps/paper_figures/`):
- `degradation_vs_prevalence.png` — scatter: prevalence vs AUC drop, with Spearman r
- `iou_vs_prevalence.png` — scatter: prevalence vs IoU alignment
- `token_mask_grid_{label}.png` — Grad-CAM vs mask comparison per focal pathology
- `token_mask_overlay_{label}.png` — heatmap + surviving token overlay

In [ ]:
# ── Cell 1 — Imports + paths ──────────────────────────────────────────────────
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from PIL import Image
from scipy import stats

from src.config import CKPT_DIR, SPLITS_DIR
from src.dataset import LABEL_COLS

PROJECT_ROOT = Path('..').resolve()
MAPS_DIR     = PROJECT_ROOT / 'maps'
FIG_DIR      = MAPS_DIR / 'paper_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RARE_LABELS  = ['Pneumonia', 'Fracture', 'Lung Lesion', 'Pleural Other']
RELIABLE_MIN = 5

# Focal pathologies for the visualization section
FOCAL = ['Pneumonia', 'Pleural Effusion', 'Cardiomegaly']

# Validation split (for positive image selection)
df_valid = pd.read_csv(SPLITS_DIR / 'valid_frontal.csv')
val_pos  = df_valid[LABEL_COLS].sum().to_dict()

print('Project root:', PROJECT_ROOT)
print('Paper figures will be saved to:', FIG_DIR)

---
## Section A — Per-label degradation vs prevalence

In [ ]:
# ── Cell 2 — Training prevalence (computed from local train_full.csv) ──────────
#
# Uses the frontal training split directly — no Drive download needed.
# Uncertain labels (-1) treated same as component 1 (U-ones/U-zeros policy
# from dataset.py) to match actual training prevalence.

from src.dataset import remap_uncertain

train_csv = SPLITS_DIR / 'train_full.csv'
if not train_csv.exists():
    raise FileNotFoundError('train_full.csv not found — check SPLITS_DIR')

df_train = pd.read_csv(train_csv)
df_train = remap_uncertain(df_train, LABEL_COLS)

PREVALENCE = {}
for l in LABEL_COLS:
    if l in df_train.columns:
        PREVALENCE[l] = float((df_train[l] == 1.0).mean() * 100)

print('Training prevalence (after uncertain remapping):')
for l, p in sorted(PREVALENCE.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(p / 2)
    print(f'  {l:<32} {p:5.1f}%  {bar}')

In [ ]:
# ── Cell 3 — Load AUC results from saved CSVs ─────────────────────────────────
#
# Loads per-label AUC from notebooks 04, 05, 06.
# If a CSV is missing, that method gets NaN and a warning is printed.

def safe_load(csv_path, index_col='Label'):
    p = Path(csv_path)
    if not p.exists():
        print(f'  [MISSING] {p.name} — download from Drive')
        return pd.DataFrame()
    return pd.read_csv(p, index_col=index_col)

df_head  = safe_load(CKPT_DIR / 'pruning_auc_comparison.csv')
df_token = safe_load(CKPT_DIR / 'token_pruning_auc.csv')
df_ca    = safe_load(CKPT_DIR / 'ca_vs_standard_auc.csv')

# Helper: get AUC for a label from a dataframe column
def get_auc(df, label, col):
    if df.empty or label not in df.index or col not in df.columns:
        return float('nan')
    return float(df.loc[label, col])

# Build unified results dict per label
AUC = {}
for l in LABEL_COLS:
    AUC[l] = {
        'baseline':    get_auc(df_head,  l, 'baseline'),
        'head_25':     get_auc(df_head,  l, '25pct'),
        'head_50':     get_auc(df_head,  l, '50pct'),
        'head_75':     get_auc(df_head,  l, '75pct'),
        'std_keep70':  get_auc(df_token, l, 'dynvit_70pct'),
        'std_keep50':  get_auc(df_token, l, 'dynvit_50pct'),
        'std_keep30':  get_auc(df_token, l, 'dynvit_30pct'),
        'ca_keep70':   get_auc(df_ca,    l, 'ca_70'),
        'ca_keep50':   get_auc(df_ca,    l, 'ca_50'),
        'ca_keep30':   get_auc(df_ca,    l, 'ca_30'),
    }

print('AUC data loaded. Reliable labels (Val+ ≥ 5):')
reliable = [l for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]
for l in reliable:
    base = AUC[l]['baseline']
    s30  = AUC[l]['std_keep30']
    c30  = AUC[l]['ca_keep30']
    print(f'  {l:<32}  base={base:.3f}  std30={s30:.3f}  ca30={c30:.3f}  '
          f'Δstd={s30-base:+.3f}  Δca={c30-base:+.3f}')

In [ ]:
# ── Cell 4 — Figure A: Degradation vs prevalence scatter ──────────────────────
#
# Only annotate notable points to avoid clutter:
#   - All rare labels (bold)
#   - Top 2 by degradation in each series
#   - Labels where std and CA diverge most
# p-value included in legend.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (keep_suffix, label_suffix) in enumerate([
    ('30', '30% keep (most aggressive)'),
    ('70', '70% keep (least aggressive)'),
]):
    ax = axes[ax_idx]

    prevs, std_drops, ca_drops, names = [], [], [], []
    for l in reliable:
        base = AUC[l]['baseline']
        std  = AUC[l][f'std_keep{keep_suffix}']
        ca   = AUC[l][f'ca_keep{keep_suffix}']
        if np.isnan(base) or np.isnan(std) or np.isnan(ca):
            continue
        prevs.append(PREVALENCE.get(l, float('nan')))
        std_drops.append(base - std)
        ca_drops.append(base - ca)
        names.append(l)

    prevs     = np.array(prevs)
    std_drops = np.array(std_drops)
    ca_drops  = np.array(ca_drops)

    r_std, p_std = stats.spearmanr(prevs, std_drops)
    r_ca,  p_ca  = stats.spearmanr(prevs, ca_drops)

    def fmt_p(p):
        return 'p<0.001' if p < 0.001 else f'p={p:.3f}'

    ax.scatter(prevs, std_drops, color='steelblue', s=90, zorder=3,
               label=f'Standard DynamicViT  r={r_std:.2f} ({fmt_p(p_std)})')
    ax.scatter(prevs, ca_drops,  color='coral',     s=90, marker='^', zorder=3,
               label=f'CA-DynamicViT         r={r_ca:.2f} ({fmt_p(p_ca)})')

    for drops, color in [(std_drops, 'steelblue'), (ca_drops, 'coral')]:
        m, b = np.polyfit(prevs, drops, 1)
        xs   = np.linspace(prevs.min(), prevs.max(), 100)
        ax.plot(xs, m * xs + b, color=color, linewidth=1.5, linestyle='--', alpha=0.6)

    # Annotate only: rare labels + top 2 most-degraded std + most divergent
    std_order   = np.argsort(std_drops)[::-1]
    divergence  = np.abs(std_drops - ca_drops)
    div_order   = np.argsort(divergence)[::-1]
    annotate_idx = set()
    for i, n in enumerate(names):
        if n in RARE_LABELS:
            annotate_idx.add(i)
    for i in std_order[:2]:
        annotate_idx.add(i)
    for i in div_order[:2]:
        annotate_idx.add(i)

    SHORT = {
        'Enlarged Cardiomediastinum': 'Enl.CM',
        'Support Devices':            'Supp.Dev',
        'Pleural Effusion':           'Pl.Eff',
        'Pleural Other':              'Pl.Other',
        'No Finding':                 'No Find.',
    }
    for i in annotate_idx:
        name   = names[i]
        short  = SHORT.get(name, name)
        is_rare = name in RARE_LABELS
        # Annotate std point
        ax.annotate(short, (prevs[i], std_drops[i]),
                    fontsize=7.5, color='steelblue',
                    fontweight='bold' if is_rare else 'normal',
                    xytext=(5, 3), textcoords='offset points')

    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Training prevalence (%)', fontsize=11)
    ax.set_ylabel('AUC degradation (baseline − pruned)', fontsize=11)
    ax.set_title(f'Degradation vs prevalence — {label_suffix}', fontsize=11)
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.3)

    print(f'Keep {keep_suffix}%: r(std)={r_std:.3f} {fmt_p(p_std)} | '
          f'r(CA)={r_ca:.3f} {fmt_p(p_ca)}')

plt.tight_layout()
plt.savefig(FIG_DIR / 'degradation_vs_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved degradation_vs_prevalence.png')

In [ ]:
# ── Cell 5 — Figure A2: IoU vs prevalence ─────────────────────────────────────
#
# Fixes vs previous version:
#   - Different markers: circle (std) vs triangle (CA) — distinguishable without colour
#   - Larger points, higher alpha for standard (was hidden behind CA)
#   - Only annotate rare labels + top/bottom 2 outliers per series
#   - p-value in legend
#   - Wider figure to reduce label overlap

df_iou_std = safe_load(CKPT_DIR / 'token_pruning_iou.csv', index_col=None)
df_iou_ca  = safe_load(CKPT_DIR / 'ca_vs_standard_iou.csv', index_col=None)

if df_iou_std.empty or df_iou_ca.empty:
    print('IoU CSVs not available — skipping this figure.')
else:
    fig, ax = plt.subplots(figsize=(11, 6))

    SHORT = {
        'Enlarged Cardiomediastinum': 'Enl.CM',
        'Support Devices':            'Supp.Dev',
        'Pleural Effusion':           'Pl.Eff',
        'No Finding':                 'No Find.',
    }

    series_data = {}

    for df_iou, color, marker, label_prefix, variant in [
        (df_iou_std[df_iou_std['keep_ratio'] == 30],
         'steelblue', 'o', 'Standard DynamicViT', 'std'),
        (df_iou_ca[ df_iou_ca['keep_ratio']  == 30],
         'coral',     '^', 'CA-DynamicViT',        'ca'),
    ]:
        prevs, ious, names = [], [], []
        for _, row in df_iou.iterrows():
            l = row['label']
            if l not in PREVALENCE or int(val_pos.get(l, 0)) < RELIABLE_MIN:
                continue
            prevs.append(PREVALENCE[l])
            ious.append(row['mean_iou'])
            names.append(l)

        if not prevs:
            continue
        prevs = np.array(prevs)
        ious  = np.array(ious)
        r, p  = stats.spearmanr(prevs, ious)
        pstr  = 'p<0.001' if p < 0.001 else f'p={p:.3f}'

        series_data[variant] = (prevs, ious, names)

        ax.scatter(prevs, ious, color=color, marker=marker, s=110,
                   zorder=4, alpha=0.9,
                   label=f'{label_prefix}  r={r:.2f} ({pstr})')
        m, b = np.polyfit(prevs, ious, 1)
        xs   = np.linspace(prevs.min(), prevs.max(), 100)
        ax.plot(xs, m*xs+b, color=color, linewidth=1.8,
                linestyle='--', alpha=0.65)

        # Annotate: rare labels + 2 highest IoU + 2 lowest IoU
        iou_order    = np.argsort(ious)
        annotate_idx = set()
        for i, n in enumerate(names):
            if n in RARE_LABELS:
                annotate_idx.add(i)
        for i in list(iou_order[-2:]) + list(iou_order[:2]):
            annotate_idx.add(i)

        offsets = {'std': (5,  4), 'ca': (5, -10)}
        for i in annotate_idx:
            name  = names[i]
            short = SHORT.get(name, name)
            bold  = name in RARE_LABELS
            dx, dy = offsets[variant]
            ax.annotate(short, (prevs[i], ious[i]),
                        fontsize=7.5, color=color,
                        fontweight='bold' if bold else 'normal',
                        xytext=(dx, dy), textcoords='offset points')

    ax.set_xlabel('Training prevalence (%)', fontsize=12)
    ax.set_ylabel('Mean IoU (token mask vs Grad-CAM)', fontsize=12)
    ax.set_title('Spatial alignment vs training prevalence — 30% keep\n'
                 'Standard DynamicViT favours common labels (r=0.65); '
                 'CA-DynamicViT equalises (r=0.17)', fontsize=11)
    ax.set_ylim(0, 0.8)
    ax.legend(fontsize=9.5)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'iou_vs_prevalence.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved iou_vs_prevalence.png')

---
## Section B — Grad-CAM vs token mask visualization

In [ ]:
# ── Cell 6 — Load maps ────────────────────────────────────────────────────────

MASKS_DIR = MAPS_DIR / 'token_masks'

def load_npy(path, name):
    p = Path(path)
    if not p.exists():
        print(f'  [MISSING] {name} — download from Drive: {p.name}')
        return None
    arr = np.load(p)
    print(f'  Loaded {name}: {arr.shape}')
    return arr

# Grad-CAM: [202, 14, 14, 14] = [n_val, n_labels, 14, 14]
gcam = load_npy(MAPS_DIR / 'gradcam' / 'gradcam_maps.npy', 'Grad-CAM baseline')

# Token masks: [202, 14, 14]
masks = {}
for key, fname in [
    ('std_30',  'masks_30pct.npy'),
    ('std_70',  'masks_70pct.npy'),
    ('ca_30',   'masks_ca_30pct.npy'),
    ('ca_70',   'masks_ca_70pct.npy'),
]:
    m = load_npy(MASKS_DIR / fname, key)
    if m is not None:
        masks[key] = m

if gcam is None or len(masks) < 4:
    print('\n⚠ Download missing files from Drive before running Section B.')
else:
    print('\nAll maps loaded — ready for visualization.')

In [ ]:
# ── Cell 7 — Image helper functions ──────────────────────────────────────────

IMAGE_ROOT = PROJECT_ROOT

def load_img_rgb(path):
    """Load a chest X-ray as RGB uint8 [H, W, 3]."""
    img = np.array(Image.open(IMAGE_ROOT / path).convert('RGB'))
    return img

def resize_to(arr_2d, h, w, interp=cv2.INTER_LINEAR):
    return cv2.resize(arr_2d.astype(np.float32), (w, h), interpolation=interp)

def gradcam_overlay(img_rgb, cam_14, alpha=0.45):
    """Jet heatmap of cam_14 blended over img_rgb."""
    h, w = img_rgb.shape[:2]
    cam  = resize_to(cam_14, h, w)
    cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    hm   = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    hm   = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
    return np.clip(alpha * hm + (1 - alpha) * img_rgb, 0, 255).astype(np.uint8)

def token_mask_overlay(img_rgb, mask_14, alpha=0.45):
    """
    Green overlay where tokens are kept, red where dropped.
    Uses nearest-neighbour to preserve the 14×14 block structure.
    """
    h, w = img_rgb.shape[:2]
    mask = resize_to(mask_14, h, w, cv2.INTER_NEAREST)
    canvas = img_rgb.copy().astype(np.float32)
    kept   = mask > 0.5
    dropped= ~kept
    canvas[kept]    = alpha * np.array([60, 200, 60])   + (1-alpha) * canvas[kept]
    canvas[dropped] = alpha * np.array([220, 50, 50])   + (1-alpha) * canvas[dropped]
    return canvas.astype(np.uint8)

def gradcam_plus_mask_overlay(img_rgb, cam_14, mask_14, alpha_cam=0.35, alpha_mask=0.35):
    """
    Combined: Grad-CAM heatmap where tokens are kept (green border), greyed out where dropped.
    Highlights agreement/disagreement between saliency and survival.
    """
    h, w = img_rgb.shape[:2]
    cam  = resize_to(cam_14, h, w)
    cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    hm   = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    hm   = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB).astype(np.float32)
    mask = resize_to(mask_14, h, w, cv2.INTER_NEAREST)
    base = img_rgb.astype(np.float32)
    # Dropped tokens: grey out the Grad-CAM
    result = alpha_cam * hm + (1 - alpha_cam) * base
    grey   = np.full_like(base, 128)
    result[mask < 0.5] = (0.7 * grey + 0.3 * base)[mask < 0.5]
    return result.astype(np.uint8)

def pick_best_images(label, n=3, score_by='gcam'):
    """
    For a given label, pick the n positive validation images with the
    highest mean Grad-CAM activation (clearest diagnostic signal).
    Returns list of (img_idx, path) tuples.
    """
    li       = LABEL_COLS.index(label)
    pos_mask = (df_valid[label] == 1.0).values
    pos_idx  = np.where(pos_mask)[0]
    if gcam is not None and score_by == 'gcam':
        scores = [gcam[i, li].mean() for i in pos_idx]
        order  = np.argsort(scores)[::-1][:n]
        selected = pos_idx[order]
    else:
        selected = pos_idx[:n]
    return [(int(i), df_valid.iloc[i]['Path']) for i in selected]

print('Helper functions defined.')

In [ ]:
# ── Cell 8 — Main paper figure: Grad-CAM vs token masks ──────────────────────
#
# Grid layout per focal pathology:
#   Rows    = images (up to 3 per pathology)
#   Columns = Original | Grad-CAM | Std 30% mask | CA 30% mask | Std 70% mask
#
# The key story column-wise:
#   Col 3 (Std 30%): drops diagnostic region for rare pathologies
#   Col 4 (CA 30%):  preserves diagnostic region
#   Col 5 (Std 70%): demonstrates non-monotonic failure (worse than CA 30%)

if gcam is None or not masks:
    print('Maps not loaded — run Cell 6 first.')
else:
    N_COLS = 5
    COL_TITLES = [
        'Original',
        'Grad-CAM\n(baseline)',
        'Std DynViT\n30% keep',
        'CA-DynViT\n30% keep',
        'Std DynViT\n70% keep',
    ]

    for label in FOCAL:
        candidates = pick_best_images(label, n=3)
        if not candidates:
            print(f'No positive images for {label}')
            continue

        li     = LABEL_COLS.index(label)
        n_rows = len(candidates)
        fig, axes = plt.subplots(n_rows, N_COLS,
                                  figsize=(N_COLS * 3.2, n_rows * 3.2))
        if n_rows == 1:
            axes = axes[np.newaxis, :]

        for row, (img_idx, img_path) in enumerate(candidates):
            img = load_img_rgb(img_path)

            panels = [
                img,
                gradcam_overlay(img, gcam[img_idx, li]),
                token_mask_overlay(img, masks.get('std_30', np.ones((14,14)))[img_idx]),
                token_mask_overlay(img, masks.get('ca_30',  np.ones((14,14)))[img_idx]),
                token_mask_overlay(img, masks.get('std_70', np.ones((14,14)))[img_idx]),
            ]

            for col, panel in enumerate(panels):
                ax = axes[row, col]
                ax.imshow(panel, cmap='gray' if panel.ndim == 2 else None)
                ax.axis('off')
                if row == 0:
                    ax.set_title(COL_TITLES[col], fontsize=9, pad=4)
            axes[row, 0].set_ylabel(f'Image {row+1}', fontsize=8, labelpad=4)

        # Legend patches for token mask columns
        kept_patch    = mpatches.Patch(color=(60/255, 200/255, 60/255), label='Token kept')
        dropped_patch = mpatches.Patch(color=(220/255, 50/255, 50/255), label='Token dropped')
        fig.legend(handles=[kept_patch, dropped_patch],
                   loc='lower center', ncol=2, fontsize=8,
                   bbox_to_anchor=(0.5, -0.02))

        safe_name = label.replace(' ', '_').replace('/', '_')
        fig.suptitle(f'{label}  —  Grad-CAM vs token survival masks',
                     fontsize=11, y=1.01)
        plt.tight_layout()
        fname = FIG_DIR / f'token_mask_grid_{safe_name}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved {fname.name}')

In [ ]:
# ── Cell 9 — Alignment overlay: Grad-CAM + surviving tokens ──────────────────
#
# For each focal pathology, one representative image.
# Shows the Grad-CAM heatmap faded out where tokens were dropped —
# makes it visually obvious when the model drops the salient region.
#
# 2 rows × 3 cols:
#   Row 1: Standard DynamicViT 30% (Grad-CAM dimmed where tokens dropped)
#   Row 2: CA-DynamicViT 30%

if gcam is None or not masks:
    print('Maps not loaded — run Cell 6 first.')
else:
    n_focal = len(FOCAL)
    fig, axes = plt.subplots(2, n_focal, figsize=(n_focal * 3.5, 7))

    row_titles = ['Std DynViT 30% keep', 'CA-DynViT 30% keep']

    for col, label in enumerate(FOCAL):
        li         = LABEL_COLS.index(label)
        candidates = pick_best_images(label, n=1)
        if not candidates:
            continue
        img_idx, img_path = candidates[0]
        img = load_img_rgb(img_path)

        for row, mask_key in enumerate(['std_30', 'ca_30']):
            ax  = axes[row, col]
            overlay = gradcam_plus_mask_overlay(
                img, gcam[img_idx, li],
                masks.get(mask_key, np.ones((14,14)))[img_idx]
            )
            ax.imshow(overlay)
            ax.axis('off')
            if row == 0:
                ax.set_title(label, fontsize=10, pad=4)
            if col == 0:
                ax.set_ylabel(row_titles[row], fontsize=9)

    fig.suptitle('Grad-CAM regions surviving token pruning\n'
                 '(bright = kept + salient, grey = dropped)',
                 fontsize=11)
    plt.tight_layout()
    fname = FIG_DIR / 'token_mask_overlay_comparison.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname.name}')

In [ ]:
# ── Cell 10 — Pneumonia deep-dive (paper key figure) ─────────────────────────
#
# Top 3 Pneumonia images (by Grad-CAM activation score).
# 4 columns only: Original | Grad-CAM | Std 30% | CA 30%
# Larger panels, cleaner for a paper page.

if gcam is None or not masks:
    print('Maps not loaded — run Cell 6 first.')
else:
    label      = 'Pneumonia'
    li         = LABEL_COLS.index(label)
    candidates = pick_best_images(label, n=3)   # top 3 by Grad-CAM score

    N_COLS     = 4
    col_titles = ['Original', 'Grad-CAM\n(baseline)', 'Std DynViT\n30% keep', 'CA-DynViT\n30% keep']
    mask_keys  = ['std_30', 'ca_30']

    n_rows = len(candidates)
    fig, axes = plt.subplots(n_rows, N_COLS,
                              figsize=(N_COLS * 3.6, n_rows * 3.6))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row, (img_idx, img_path) in enumerate(candidates):
        img    = load_img_rgb(img_path)
        panels = [
            img,
            gradcam_overlay(img, gcam[img_idx, li]),
            token_mask_overlay(img, masks['std_30'][img_idx]),
            token_mask_overlay(img, masks['ca_30'][img_idx]),
        ]
        for col, panel in enumerate(panels):
            ax = axes[row, col]
            ax.imshow(panel)
            ax.axis('off')
            if row == 0:
                ax.set_title(col_titles[col], fontsize=11, pad=5)
        axes[row, 0].set_ylabel(f'Image {row+1}', fontsize=9, labelpad=5)

    kept_patch    = mpatches.Patch(color=(60/255, 200/255, 60/255), label='Token kept')
    dropped_patch = mpatches.Patch(color=(220/255, 50/255,  50/255), label='Token dropped')
    fig.legend(handles=[kept_patch, dropped_patch],
               loc='lower center', ncol=2, fontsize=10,
               bbox_to_anchor=(0.5, -0.02))

    fig.suptitle('Pneumonia — top 3 validation images by Grad-CAM saliency\n'
                 'Standard DynViT 30% drops the salient region; '
                 'CA-DynViT 30% preserves it',
                 fontsize=11, y=1.02)
    plt.tight_layout()
    fname = FIG_DIR / 'pneumonia_deepdive.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname.name}')